# Nén các thư mục trong Extracted thành file .zip

Notebook này chạy trên **tài khoản A** (chủ sở hữu dataset). Với mỗi thư mục con trong
`/content/drive/MyDrive/aic26/Extracted/` (K01…K20, L21…L30, ...), notebook tạo một file
`.zip` tương ứng và lưu vào `/content/drive/MyDrive/aic26/Zipped/`.

**Đặc điểm:**
- **Nén cục bộ trước, copy lên Drive sau** → nhanh và ổn định hơn ghi thẳng vào Drive.
- **Chống tràn ổ Colab**: trước khi nén mỗi folder, notebook kiểm tra dung lượng trống. Nếu
  không đủ chỗ, nó **DỪNG ngay và báo lỗi rõ ràng** (thay vì crash giữa chừng).
- **Resumable**: folder nào đã có `.zip` sẽ được bỏ qua → mất kết nối rồi chạy lại vẫn tiếp tục.
- **Mặc định `ZIP_STORED` (không nén lại)** vì ảnh/video đã nén sẵn → chỉ gom file, nhanh hơn nhiều.

In [ ]:
# 1. Mount Google Drive (dang nhap bang tai khoan A - chu so huu dataset)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Cau hinh
from pathlib import Path
import zipfile

SRC_DIR = Path('/content/drive/MyDrive/aic26/Extracted')   # thu muc chua K01, K02, ...
OUT_DIR = Path('/content/drive/MyDrive/aic26/Zipped')      # noi luu file .zip tren Drive
LOCAL_TMP = Path('/content/zip_tmp')                       # thu muc tam tren o cuc bo Colab

# --- Cau hinh chong tran o ---
MIN_FREE_GB = 5.0          # luon chua lai it nhat bay nhieu GB tren o cuc bo
SIZE_MARGIN = 1.05         # du phong 5% (zip co the hoi lon hon tong file mot chut)

# Neu True: folder nao qua lon so voi o cuc bo se duoc nen THANG len Drive (khong dung o tam).
#   -> chay duoc voi folder lon hon o Colab, nhung cham hon.
# Neu False: gap folder qua lon se DUNG han va bao loi.
AUTO_FALLBACK_TO_DRIVE = False

# Khong nen lai (ZIP_STORED) vi media da nen san. Doi thanh zipfile.ZIP_DEFLATED neu
# thu muc chua file de nen (txt, json...) va ban muon giam dung luong.
COMPRESSION = zipfile.ZIP_STORED

OUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_TMP.mkdir(parents=True, exist_ok=True)
assert SRC_DIR.is_dir(), f'Khong tim thay thu muc nguon: {SRC_DIR}'

In [ ]:
# 3. Xac dinh danh sach thu muc can nen
# Tu dong quet moi thu muc con trong SRC_DIR (bo qua file le nhu .done_L30).
folders = sorted([p for p in SRC_DIR.iterdir() if p.is_dir()], key=lambda p: p.name)

# (Tuy chon) Neu chi muon nen dung K01..K20 va L21..L30, bo comment 2 dong duoi:
# wanted = {f'K{i:02d}' for i in range(1, 21)} | {f'L{i:02d}' for i in range(21, 31)}
# folders = [p for p in folders if p.name in wanted]

print(f'Se nen {len(folders)} thu muc:')
for p in folders:
    print(' -', p.name)

In [ ]:
# 4. Cac ham tien ich: nen folder, do dung luong, kiem tra o trong
import shutil

def human(nbytes):
    n = float(nbytes)
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if n < 1024:
            return f'{n:.1f}{unit}'
        n /= 1024
    return f'{n:.1f}PB'

def folder_size(folder: Path) -> int:
    """Tong dung luong (byte) cua toan bo file trong folder."""
    return sum(p.stat().st_size for p in folder.rglob('*') if p.is_file())

def free_bytes(path='/content') -> int:
    """Dung luong trong con lai tren o chua `path` (o cuc bo Colab)."""
    return shutil.disk_usage(path).free

def zip_folder(folder: Path, zip_path: Path, compression=zipfile.ZIP_STORED):
    """Nen toan bo file trong `folder` vao `zip_path`. Giu ten thu muc goc ben trong zip."""
    files = [p for p in folder.rglob('*') if p.is_file()]
    with zipfile.ZipFile(zip_path, 'w', compression=compression, allowZip64=True) as zf:
        for p in files:
            arcname = p.relative_to(folder.parent)   # vd 'K01/frame_0001.jpg'
            zf.write(p, arcname)
    return len(files)

In [ ]:
# 5. Vong lap nen - CO KIEM TRA CHONG TRAN O
import time

min_free = MIN_FREE_GB * 1024**3

for idx, folder in enumerate(folders, 1):
    name = folder.name
    final_zip = OUT_DIR / f'{name}.zip'
    tmp_zip = LOCAL_TMP / f'{name}.zip'

    if final_zip.exists():
        print(f'[{idx}/{len(folders)}] BO QUA {name} (da co {final_zip.name})')
        continue

    # --- Kiem tra dung luong TRUOC khi nen ---
    est = folder_size(folder) * SIZE_MARGIN     # zip du kien ~ tong file (ZIP_STORED)
    free = free_bytes('/content')
    need = est + min_free                        # can du cho + con chua lai min_free

    if free < need:
        if AUTO_FALLBACK_TO_DRIVE:
            # Folder qua lon -> nen thang len Drive (khong dung o tam cuc bo)
            print(f'[{idx}/{len(folders)}] {name}: o cuc bo khong du '
                  f'(can ~{human(est)}, trong {human(free)}). '
                  f'--> nen THANG len Drive.', flush=True)
            t0 = time.time()
            n_files = zip_folder(folder, final_zip, COMPRESSION)
            print(f'        -> xong (tren Drive): {n_files} file, '
                  f'{human(final_zip.stat().st_size)}, {time.time()-t0:.1f}s', flush=True)
            continue
        else:
            # DUNG HAN va bao loi ro rang
            msg = (
                '\n' + '=' * 70 + '\n'
                'DUNG: KHONG DU DUNG LUONG TREN O CUC BO COLAB\n'
                + '=' * 70 + '\n'
                f'  Thu muc dang xu ly : {name}\n'
                f'  Kich thuoc uoc tinh: {human(est)}\n'
                f'  O cuc bo con trong : {human(free)}\n'
                f'  Can toi thieu      : {human(need)} '
                f'(gom {human(est)} + du phong {MIN_FREE_GB}GB)\n'
                + '-' * 70 + '\n'
                '  Cach xu ly:\n'
                '   1) Dat AUTO_FALLBACK_TO_DRIVE = True o cell 2 de nen thang len Drive, HOAC\n'
                '   2) Chia nho thu muc nay ra roi nen tung phan, HOAC\n'
                '   3) Dung runtime co o dia lon hon.\n'
                + '=' * 70
            )
            print(msg, flush=True)
            raise RuntimeError(f'Het cho tren o cuc bo khi xu ly {name} '
                               f'(can {human(est)}, trong {human(free)}).')

    # --- Du cho: nen cuc bo roi chuyen len Drive ---
    t0 = time.time()
    print(f'[{idx}/{len(folders)}] Dang nen {name} (~{human(est)}, o trong {human(free)}) ...',
          flush=True)
    n_files = zip_folder(folder, tmp_zip, COMPRESSION)
    size = tmp_zip.stat().st_size
    shutil.move(str(tmp_zip), str(final_zip))
    print(f'        -> xong: {n_files} file, {human(size)}, {time.time()-t0:.1f}s -> {final_zip}',
          flush=True)

print('\nHOAN TAT tat ca thu muc.')

In [ ]:
# 6. (Tuy chon) Kiem tra lai cac file .zip da tao tren Drive
zips = sorted(OUT_DIR.glob('*.zip'), key=lambda p: p.name)
print(f'Co {len(zips)} file zip trong {OUT_DIR}:')
for z in zips:
    print(f' - {z.name:12s} {human(z.stat().st_size)}')

In [ ]:
# 7. Dem so anh trong folder
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.gif', '.tif', '.tiff'}

def count_images(folder: Path) -> int:
    """Dem so file anh trong folder (tinh ca thu muc con)."""
    return sum(1 for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXTS)

# --- (a) Dem trong 1 folder cu the ---
ONE_FOLDER = Path('/content/drive/MyDrive/aic26/Extracted/K01')
print(f'{ONE_FOLDER.name}: {count_images(ONE_FOLDER):,} anh')

# --- (b) (Tuy chon) Dem cho TAT CA folder trong Extracted ---
print('\nSo anh tung folder:')
total = 0
for folder in sorted([p for p in SRC_DIR.iterdir() if p.is_dir()], key=lambda p: p.name):
    n = count_images(folder)
    total += n
    print(f' - {folder.name:12s} {n:>8,} anh')
print(f'\nTONG CONG: {total:,} anh')
